# Reranker ONNX export on Colab (DEC-062)

Madhya Kanoon Saathi, W1.5. Converts `BAAI/bge-reranker-v2-m3` (Apache-2.0) to ONNX **once**, so the laptop and
the Lambda run it with onnxruntime (no torch). Runtime: **CPU is enough** (GPU not needed); High-RAM helps.

What it does:
1. downloads the model at the pinned revision (public, no token needed);
2. exports to ONNX (inputs `input_ids`, `attention_mask`; output `logits`);
3. **checks** the ONNX scores equal the PyTorch scores on English, Hindi and Hinglish pairs;
4. writes `export_info.json` (versions, revision, sha256 of every file) and saves all to Drive `mks_reranker/`.

Then download the folder `mks_reranker/` and put it at `data/models/bge-reranker-v2-m3/` in the repo.

In [ ]:
!pip -q install onnx onnxruntime
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, pathlib, platform, time
import numpy as np, torch, transformers, onnx, onnxruntime as ort
from transformers import AutoTokenizer, AutoModelForSequenceClassification
REPO, REV = 'BAAI/bge-reranker-v2-m3', '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
OUT = pathlib.Path('/content/drive/MyDrive/mks_reranker'); OUT.mkdir(exist_ok=True)
print('torch', torch.__version__, '| transformers', transformers.__version__, '| onnx', onnx.__version__, '| ort', ort.__version__)

In [ ]:
tok = AutoTokenizer.from_pretrained(REPO, revision=REV)
model = AutoModelForSequenceClassification.from_pretrained(REPO, revision=REV).eval()
PAIRS = [
    ('who is entitled to free legal aid', 'Every person who has to file or defend a case shall be entitled to legal services under this Act if that person is a member of a Scheduled Caste or Scheduled Tribe.'),
    ('सूचना का अधिकार में व्यक्तिगत जानकारी', 'information which relates to personal information the disclosure of which has no relationship to any public activity or interest'),
    ('eshram registration ke liye umar kitni honi chahiye', 'any individual engaged in unorganised work and aged 16 years and above is eligible to register on the eShram portal'),
    ('minimum wages for unskilled workers', 'The Rent Authority shall dispose of the application within sixty days.'),
]
enc = tok([q for q, _ in PAIRS], [p for _, p in PAIRS], padding=True, truncation=True, max_length=512, return_tensors='pt')
with torch.no_grad():
    ref = model(**enc).logits[:, 0].numpy()
print('torch scores', np.round(ref, 4))

In [ ]:
tmp = pathlib.Path('/content/export'); tmp.mkdir(exist_ok=True)
t0 = time.time()
torch.onnx.export(
    model, (enc['input_ids'], enc['attention_mask']), str(tmp / 'model.onnx'),
    input_names=['input_ids', 'attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'}, 'logits': {0: 'batch'}},
    opset_version=17, dynamo=False,
)
# over 2 GB: save a clean pair on Drive, model.onnx + one external data file
onnx_path = OUT / 'model.onnx'
onnx.save_model(onnx.load(str(tmp / 'model.onnx')), str(onnx_path), save_as_external_data=True,
                all_tensors_to_one_file=True, location='model.onnx.data', size_threshold=1024)
print('exported in', round(time.time() - t0), 's;', sorted(p.name for p in OUT.iterdir()))

In [ ]:
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
got = sess.run(['logits'], {'input_ids': enc['input_ids'].numpy(), 'attention_mask': enc['attention_mask'].numpy()})[0][:, 0]
diff = float(np.abs(got - ref).max())
print('onnx scores ', np.round(got, 4), '| max abs diff vs torch', diff)
assert diff < 1e-3, 'ONNX export does not match PyTorch: do not use these files'
assert got[0] > got[3] and got[2] > got[3], 'relevant pairs should outscore the unrelated pair'

In [ ]:
tok.save_pretrained(str(OUT))
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
info = {
    'repo': REPO, 'revision': REV, 'licence': 'Apache-2.0', 'exported': 'torch.onnx.export opset 17 (TorchScript exporter)',
    'versions': {'torch': torch.__version__, 'transformers': transformers.__version__, 'onnx': onnx.__version__,
                 'onnxruntime': ort.__version__, 'python': platform.python_version()},
    'check': {'pairs': len(PAIRS), 'max_abs_diff_vs_torch': diff, 'torch_scores': ref.tolist(), 'onnx_scores': got.tolist()},
    'files': [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha(p)} for p in sorted(OUT.iterdir()) if p.is_file() and p.name != 'export_info.json'],
}
(OUT / 'export_info.json').write_text(json.dumps(info, indent=2))
print(json.dumps(info['files'], indent=1))